# Amazon2014 Explanation Generation — `user_item_proto`

Generates TSNE plots, top-K items per prototype, and weight visualization for the best trial from the `user_item_proto` hyperopt run on amazon2014.

**Notebook pattern (modular pipeline):** This notebook drives `utilities/explanations/` — the same modular pipeline that `run_combo.py` auto-invokes at the end of every explainable training run. New runs (e.g., the ml-1m `user_item_proto` queued on LEO5) save explanations to `Master/experiments/results/<combo>/explanations/` automatically; for older runs that only left a `ray_results/` directory (no structured combo dir), this notebook reconstructs the model from `ExperimentAnalysis` and then runs the pipeline against the in-memory model.

**Outputs written to `Master/experiments/replication/explanations/` (canonical names):**
- `tsne_item_prototypes.pdf`, `tsne_user_prototypes.pdf`
- `top_k_items_per_item_prototype.csv`, `top_k_items_per_user_prototype.csv`
- `weight_viz_user{uid}_user_side.pdf`, `weight_viz_user{uid}_item_side.pdf` (user_item_proto only)

In [ ]:
import argparse
import os
import sys

import pandas as pd
import torch

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO_ROOT)

from ray.tune import ExperimentAnalysis
from rec_sys.rec_sys import RecSys
from feature_extraction.feature_extractor_factories import FeatureExtractorFactory
from utilities.explanations.accessor import get_accessor
from utilities.explanations.explainers import REGISTERED_EXPLAINERS, ExplainCtx
from utilities.explanations.items_info import load_items_info

%matplotlib inline

## 1. Load best trial from `ray_results`

The amazon2014 `user_item_proto` run predates the structured combo-results format produced by `run_combo._save_combo_results`, so we read from `~/ray_results/` via `ExperimentAnalysis`. For runs that DO have a structured combo dir (anything launched via `run_combo.py`), prefer `utilities.explanations.loader.load_recsys_from_results_dir(results_dir)` instead.

In [ ]:
MODEL_TYPE = 'user_item_proto'
DATASET = 'amazon2014'

RESULT_DIR = os.path.expanduser('~/ray_results/user_item_proto_amazon2014_DE_38210573_2026-3-31_13-15-21.491510')
DATA_DIR = os.path.join(REPO_ROOT, 'data', DATASET)
OUTPUT_DIR = os.path.join(REPO_ROOT, 'Master', 'experiments', 'replication', 'explanations')
os.makedirs(OUTPUT_DIR, exist_ok=True)

analysis = ExperimentAnalysis(RESULT_DIR)
metric = 'hit_ratio@10'
best_trial = analysis.get_best_trial(metric, 'max', scope='all')
best_checkpoint = analysis.get_best_checkpoint(best_trial, metric, 'max')
checkpoint_path = os.path.join(best_checkpoint.to_directory(), 'best_model.pth')
config = argparse.Namespace(**best_trial.config)

print(f"Best val HR@10:   {best_trial.last_result['hit_ratio@10']:.4f}")
print(f"Best val NDCG@10: {best_trial.last_result['ndcg@10']:.4f}")

In [ ]:
n_users = pd.read_csv(os.path.join(DATA_DIR, 'user_ids.csv')).shape[0]
n_items = pd.read_csv(os.path.join(DATA_DIR, 'item_ids.csv')).shape[0]

user_fe, item_fe = FeatureExtractorFactory.create_models(config.ft_ext_param, n_users, n_items)
model = RecSys(
    n_users, n_items, config.rec_sys_param, user_fe, item_fe,
    config.loss_func_name, config.loss_func_aggr,
)
model.load_state_dict(torch.load(checkpoint_path, map_location='cpu'))
model.eval()
print(f"n_users={n_users}, n_items={n_items}, model_type={MODEL_TYPE}")

## 2. Run the modular explainer pipeline

Three explainers are registered: `TSNEExplainer`, `TopKItemsExplainer`, `WeightVizExplainer`. Each declares which model types it supports via `.supports(model_type)`. `user_item_proto` triggers all three.

In [ ]:
accessor = get_accessor(MODEL_TYPE, model)
items_info = load_items_info(DATASET)

ctx = ExplainCtx(
    accessor=accessor,
    items_info=items_info,
    output_dir=OUTPUT_DIR,
    tsne_sample_size=2000,
    top_k=10,
    sample_users_for_weight_viz=[42],
)

for explainer in REGISTERED_EXPLAINERS:
    if not explainer.supports(MODEL_TYPE):
        continue
    print(f"  running '{explainer.name}' ...")
    try:
        explainer.run(ctx)
    except Exception as e:
        print(f"    ⚠ '{explainer.name}' failed: {e!r}")

print(f"\nDone → {OUTPUT_DIR}")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}")

## 3. Inspect outputs

TSNE and weight-viz PDFs are best opened externally. The top-K CSVs are tabular — show them inline below.

In [ ]:
item_proto_csv = os.path.join(OUTPUT_DIR, 'top_k_items_per_item_prototype.csv')
df_item = pd.read_csv(item_proto_csv, index_col=0)
print(f"Item prototypes: {df_item['prototype'].nunique()} × top-{ctx.top_k} items each\n")
df_item.groupby('prototype').head(5)

In [ ]:
user_proto_csv = os.path.join(OUTPUT_DIR, 'top_k_items_per_user_prototype.csv')
df_user = pd.read_csv(user_proto_csv, index_col=0)
print(f"User prototypes: {df_user['prototype'].nunique()} × top-{ctx.top_k} items each")
print("(Each user prototype is interpreted by the items maximally aligned with it in user-proto space.)\n")
df_user.groupby('prototype').head(5)

In [ ]:
pdfs = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.pdf')]
print('PDF outputs (open externally):')
for f in sorted(pdfs):
    print(f"  {os.path.join(OUTPUT_DIR, f)}")